In [1]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("m2m-local-all-cores")
    .master("local[*]")              # usa todos los cores lógicos disponibles
    .config("spark.ui.port", "4040") # UI en el puerto 4040
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/20 23:04:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from time import time

t0 = time()

# 1 millón de filas (ligero)
df = spark.range(0, 1_000_000).toDF("id")

# pequeño cálculo: contar pares/impares
res = (
    df.withColumn("par", (df.id % 2 == 0).cast("int"))
      .groupBy("par")
      .count()
      .orderBy("par")
      .collect()
)

t1 = time()

print("Resultado:", res)
print(f"Tiempo: {t1 - t0:.2f} s")


Resultado: [Row(par=0, count=500000), Row(par=1, count=500000)]
Tiempo: 0.58 s


In [2]:
import random
from time import time

N = 50_000_000   # si se va de memoria, baja a 20_000_000

sc = spark.sparkContext

t0 = time()

def inside(_):
    x = random.random()
    y = random.random()
    return 1 if x*x + y*y <= 1.0 else 0

# repartimos el trabajo en muchas particiones para usar bien los cores
num_partitions = 400

rdd = sc.parallelize(range(N), numSlices=num_partitions)
count_inside = rdd.map(inside).reduce(lambda a, b: a + b)

pi_est = 4.0 * count_inside / N
t1 = time()

print(f"π ≈ {pi_est}")
print(f"Tiempo: {t1 - t0:.2f} segundos")


[Stage 0:====================================================> (386 + 12) / 400]

π ≈ 3.14173552
Tiempo: 11.74 segundos


In [ ]:
spark.stop()
